# 1 介绍

在 `assignment1各模块实现讲解` 中我们已经讲完了每一个模块具体怎么实现，并且组成了一个完整的 TransformerLM，接下来就是进行模型训练了。

# 2 准备数据

1. 训练 BPE 分词器

```bash
uv run python run_bpe_training.py
```

运行 run_bpe_training.py 读取 TinyStories 文本，开始训练 BPE 分词器，训练完成后会生成：

* `tinystories_vocab.json`：一个字典，储存了从一个 Token ID 到 Token 的映射关系。
* `tinystories_merges.txt`：纯文本文件，每一行都代表一条合并规则，记录了在 BPE 训练过程中，哪两个相邻的 Token 被合并成了一个新的 Token。

2. 将数据集转换为 Token ID

在确认上一步成功生成了两个文件后，接着运行：

```bash
uv run python tokenize_dataset.py
```

使用刚刚创建的分词器，将训练集和验证集的文本转换成模型能理解的数字ID，结束后会生成：
* `tinystories_train.npy`：Numpy 格式的二进制文件 (.npy)，内部只包含一个巨大的一维数组，数组里装满了整数（也就是 Token ID）。这个数组是将整个 TinyStoriesV2-GPT4-train.txt 训练文本，通过分词器编码后得到的结果。
* `tinystories_val.npy`：和 tinystories_train.npy 的格式完全一样，也是一个包含 Token ID 的一维 Numpy 数组。不同的是，它的数据来源于验证集文本 TinyStoriesV2-GPT4-valid.txt。

# 3 wandb (Weights & Biases) 工具

介绍一下 wandb 这个工具，可以理解为是 tensorboard 的升级版，很好用，我觉得可以了解一下，下面是 gpt的介绍：

## 一、W\&B 是什么？

**Weights & Biases** 是一个面向 **机器学习/深度学习实验跟踪、可视化、协作与管理** 的工具平台，最初由 Google 前研究员们创办。
它的目标是帮助研究者和工程师更高效地：

* **跟踪 (track)** 每次实验的超参数、训练过程、模型指标；
* **可视化 (visualize)** 模型训练中的 loss、accuracy、梯度分布等；
* **对比 (compare)** 不同实验结果；
* **协作 (collaborate)** 与团队共享训练日志和结果；
* **管理 (manage)** 数据集、模型、超参数、结果。

换句话说，W\&B 就是 **深度学习的“实验仪表盘 + GitHub for experiments”**。

---

## 二、主要功能

### 1. 实验跟踪 (Experiment Tracking)

* 训练时自动记录：

  * **loss、accuracy** 等指标；
  * **学习率 (learning rate)** 曲线；
  * **梯度/权重分布**；
  * **系统资源使用情况 (GPU、CPU、内存)**。
* 可手动 `wandb.log({"metric": value})` 添加任何自定义指标。

### 2. 超参数管理 (Hyperparameter Tracking)

* 使用 `wandb.config` 保存超参数，如 batch\_size, lr, dropout。
* 后续在 UI 中可以按超参数搜索和筛选实验。

### 3. 可视化与仪表盘 (Visualization & Dashboard)

* Web 界面展示 **交互式曲线图**（loss vs steps、accuracy vs epoch）。
* 支持绘制 **表格、图像、视频、3D 点云** 等可视化结果。
* 可以动态缩放、选择不同实验进行对比。

### 4. 实验对比 (Experiment Comparison)

* 多个实验放在同一页面对比 loss 曲线。
* 支持基于超参数的聚合分析（比如 lr=0.001 的实验 vs lr=0.01 的实验）。

### 5. Artifact 管理 (Artifacts)

* 可以把 **数据集、模型 checkpoint、结果文件** 存储成 W\&B Artifact。
* Artifact 相当于“版本化的文件存储”，类似于 Git LFS。
* 便于追踪 “某个模型是基于哪个数据集、哪个代码版本训练的”。

### 6. Sweeps (自动超参数搜索)

* 类似 Ray Tune、Optuna，W\&B 内置超参数调优模块。
* 可以配置搜索空间（lr、batch\_size、optimizer），让 W\&B 自动运行多组实验并记录结果。

### 7. 报告与协作 (Reports & Collaboration)

* 把结果整理成 **在线报告**（带图表、文字、代码）。
* 团队成员可以共享链接，类似 Google Docs，但用于 ML 实验。

### 8. 系统监控 (System Monitoring)

* 自动记录：

  * GPU/CPU 使用率
  * 显存占用
  * 硬盘 IO
  * 网络流量
* 方便调试分布式训练。

---

## 三、使用方式

### 1. 安装

```bash
pip install wandb
```

### 2. 初始化项目

```python
import wandb

wandb.init(
    project="my-cool-project",  # 项目名称
    config={                   # 超参数
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": 32
    }
)
```

### 3. 记录指标

```python
for epoch in range(10):
    train_loss = 0.5 / (epoch+1)
    wandb.log({"epoch": epoch, "train_loss": train_loss})
```

### 4. 保存模型

```python
torch.save(model.state_dict(), "model.pt")
wandb.save("model.pt")
```

### 5. 使用 Artifact

```python
artifact = wandb.Artifact("mnist-dataset", type="dataset")
artifact.add_file("train.csv")
wandb.log_artifact(artifact)
```

# 4 Sanity Check

作业指导书中建议，在进行完整训练前，先用一个 minibatch 进行过拟合测试 。这是为了验证整个训练流程（模型、优化器、数据加载）是否能正常工作。如果模型能在一个小批量上将损失降的很低，说明代码基本没有问题。

下面是我测试使用的脚本：
```bash
uv run python train.py \
    --train_data_path=/root/data/cs336/tinystories_train.npy \
    --val_data_path=/root/data/cs336/tinystories_val.npy \
    --vocab_size=10000 \
    --context_length=256 \
    --d_model=512 \
    --num_layers=4 \
    --num_heads=16 \
    --d_ff=1344 \

    --batch_size=8 \
    --max_steps=100 \
    --eval_interval=10 \
    --log_interval=1 \
    
    --learning_rate=3e-4 \
    --warmup_steps=0 \
    
    --out_dir=./checkpoints/sanity_check \
    --device=cuda \
    --wandb_log \
    --wandb_run_name="sanity-check-fixed-lr"
```

关于 train.py 我们后面讲。

# 5 正式实验

## 5.1 学习率调优（Problem learning_rate ）

1. **实验目标**

找到一个能让模型在 TinyStories 数据集上达到最佳性能（即最低验证损失）的学习率。作业指导中有讲到，最好的学习率通常位于“稳定训练的边缘” ，即 **“最大化利用大步长加速收敛，但又不至于让模型不稳定”**。

2. **实验计划**

采用**网格搜索 (Grid Search)** 的策略，系统地测试几个数量级上不同的学习率。

3. **确定搜索范围**

一个常见的学习率搜索范围是`1e-3`到`1e-4`之间。可以测试以下几个值：

  * `1e-3` (0.001)
  * `3e-4` (0.0003) 
  * `1e-4` (0.0001)
  * `3e-5` (0.00003)

4. **设置训练参数**

对于每一次正式的训练，我们需要使用更接近最终目标的参数：

  * **`batch_size`**: 我这里使用 `64`（根据自己的 GPU 显存来进行设置）。
  * [cite\_start]**`max_steps`**: 总共处理约 3.28亿个token [cite: 1125]。根据设置 (`batch_size=64`, `context_length=256`)，计算可得 `327,680,000 / (64 * 256) = 20,000` 步。
  * **`warmup_steps`**: 预热一般都设置总步数的 10%。
  * **`wandb_run_name`**: 为每次运行设置一个清晰的名称，方便在 `wandb` 中比较结果。

5. **执行训练**

```bash
LR="1e-3"
uv run python train.py \
    --train_data_path=/root/data/cs336/tinystories_train.npy \
    --val_data_path=/root/data/cs336/tinystories_val.npy \
    --vocab_size=10000 \
    --context_length=256 \
    --d_model=512 \
    --num_layers=4 \
    --num_heads=16 \
    --d_ff=1344 \
    --batch_size=64 \
    --max_steps=20000 \
    --warmup_steps=2000 \
    --learning_rate=$LR \
    --min_learning_rate=$(echo "$LR / 10" | bc -l) \
    --out_dir=./checkpoints/lr_sweep_$LR \
    --device=cuda \
    --wandb_log \
    --wandb_run_name="lr_sweep_$LR"
```

6. **观察与评估**

  * **监控 `wandb`**: 在 `wandb` 上，你可以将所有实验的验证损失 (`eval/loss`) 曲线放在同一张图里进行比较。
  * **寻找最佳点**: 寻找哪条曲线在训练结束时达到了最低点。
  * **注意发散**: 较高的学习率（如`1e-3`）可能会导致损失在训练中途突然飙升（即“发散”），这有可能就是“稳定训练的边缘”。

# 6 train.py

In [4]:
import os
import time
import argparse
from contextlib import nullcontext

import numpy as np
import torch

# import wandb 
# from src.model.transformer import TransformerLM
# from src.optim_sched import get_adamw_cls, get_lr_cosine_schedule
# from src.data import get_batch
# from src.io import save_checkpoint, load_checkpoint
# 写笔记的电脑没装，注释掉

## 6.1 参数介绍

### 1. I/O 参数

* **`--out_dir`**：训练过程中保存 checkpoint（模型和优化器状态）的输出目录。
* **`--train_data_path`**：训练数据（已经分词并保存成 `.npy` 格式）的路径。
* **`--val_data_path`**：验证数据（`.npy` 格式）的路径。
* **`--init_from`**：决定是从零开始训练（`scratch`）还是从已有的 checkpoint 继续训练（`resume`）。



### 2. Weights & Biases 日志

* **`--wandb_log`**：是否开启 W\&B 日志记录。
* **`--wandb_project`**：上传日志的 W\&B 项目名称。
* **`--wandb_run_name`**：这次运行的名字（方便在 W\&B 面板区分不同实验）。



### 3. 模型参数

* **`--vocab_size`**：词表大小（embedding 层和最后输出层的维度）。
* **`--context_length`**：输入序列的最大长度（Transformer 的上下文窗口）。
* **`--d_model`**：模型隐藏维度（token embedding 的维度，也是注意力和前馈网络的输入输出维度）。
* **`--num_layers`**：Transformer 堆叠的层数。
* **`--num_heads`**：多头注意力的头数。
* **`--d_ff`**：前馈网络（Feed-Forward）的中间层维度。
* **`--rope_theta`**：RoPE（旋转位置编码）的 θ 参数，影响正余弦频率。



### 4. 训练参数

* **`--batch_size`**：每次训练迭代使用的样本数。
* **`--max_steps`**：训练总迭代步数。
* **`--learning_rate`**：学习率的最大值（warmup 之后的峰值）。
* **`--min_learning_rate`**：余弦退火后的最小学习率。
* **`--warmup_steps`**：学习率预热步数（从 0 增长到最大值）。
* **`--weight_decay`**：权重衰减系数（L2 正则化，用来防止过拟合）。
* **`--beta1`**：AdamW 优化器的一阶动量系数。
* **`--beta2`**：AdamW 优化器的二阶动量系数。
* **`--grad_clip`**：梯度裁剪阈值，避免梯度爆炸。



### 5. 评估与日志

* **`--eval_interval`**：多少步进行一次验证（在验证集上计算 loss / perplexity）。
* **`--eval_steps`**：每次验证时在验证集上取多少 batch。
* **`--log_interval`**：多少步打印一次训练日志（loss、lr、速度）。



### 6. 性能优化

* **`--device`**：使用的设备（`cpu`、`cuda`、`mps`）。
* **`--compile`**：是否使用 `torch.compile` 加速模型。

In [1]:
def get_args():
    """解析命令行参数"""
    parser = argparse.ArgumentParser(description="Train a Transformer Language Model")

    # I/O 参数
    parser.add_argument("--out_dir", type=str, default="checkpoints", help="Output directory for checkpoints")
    parser.add_argument("--train_data_path", type=str, required=True, help="Path to tokenized training data (.npy)")
    parser.add_argument("--val_data_path", type=str, required=True, help="Path to tokenized validation data (.npy)")
    parser.add_argument("--init_from", type=str, default="scratch", choices=["scratch", "resume"], help="Start from scratch or resume from out_dir")
    
    # WandB 日志
    parser.add_argument("--wandb_log", action="store_true", help="Enable logging to Weights & Biases")
    parser.add_argument("--wandb_project", type=str, default="cs336_assignment1", help="W&B project name")
    parser.add_argument("--wandb_run_name", type=str, default=f"train-{int(time.time())}", help="W&B run name")

    # 模型参数
    parser.add_argument("--vocab_size", type=int, required=True, help="Vocabulary size")
    parser.add_argument("--context_length", type=int, default=256, help="Maximum context length")
    parser.add_argument("--d_model", type=int, default=512, help="Model dimension")
    parser.add_argument("--num_layers", type=int, default=4, help="Number of transformer layers")
    parser.add_argument("--num_heads", type=int, default=16, help="Number of attention heads")
    parser.add_argument("--d_ff", type=int, default=1344, help="Dimension of the feed-forward layer")
    parser.add_argument("--rope_theta", type=float, default=10000.0, help="Theta for RoPE")

    # 训练参数
    parser.add_argument("--batch_size", type=int, default=64, help="Batch size")
    parser.add_argument("--max_steps", type=int, default=20000, help="Total training steps")
    parser.add_argument("--learning_rate", type=float, default=3e-4, help="Maximum learning rate")
    parser.add_argument("--min_learning_rate", type=float, default=3e-5, help="Minimum learning rate")
    parser.add_argument("--warmup_steps", type=int, default=2000, help="Number of warmup steps")
    parser.add_argument("--weight_decay", type=float, default=0.1, help="Weight decay")
    parser.add_argument("--beta1", type=float, default=0.9, help="AdamW beta1")
    parser.add_argument("--beta2", type=float, default=0.95, help="AdamW beta2")
    parser.add_argument("--grad_clip", type=float, default=1.0, help="Gradient clipping value")

    # 评估与日志记录
    parser.add_argument("--eval_interval", type=int, default=250, help="Steps between evaluations")
    parser.add_argument("--eval_steps", type=int, default=100, help="Number of steps for evaluation")
    parser.add_argument("--log_interval", type=int, default=10, help="Steps between logging training loss")
    
    # 性能
    parser.add_argument("--device", type=str, default="cuda", help="Device to use ('cpu', 'cuda', 'mps')")
    parser.add_argument("--compile", action="store_true", help="Use torch.compile for performance")

    return parser.parse_args()

## 6.2 模型评估

1. **函数声明**

```python
@torch.no_grad()
def evaluate(model, data, context_length, batch_size, device, max_steps):
```

* `@torch.no_grad()`：告诉 PyTorch 在这段函数里不需要计算梯度，减少显存和计算开销。评估时我们不更新参数，所以禁用反向传播。
* 参数含义：

  * `model`：待评估的 Transformer 模型。
  * `data`：验证数据（这里是 `.npy` 内存映射的 token 序列）。
  * `context_length`：输入序列长度（窗口大小）。
  * `batch_size`：每次拿多少样本评估。
  * `device`：运行设备（cpu / cuda）。
  * `max_steps`：评估时抽多少个 batch。

2. **切换到评估模式**

```python
    model.eval()
```

* `model.eval()`：把模型切换到**评估模式**，这会关闭 dropout、切换 LayerNorm/BatchNorm 等到推理行为。

3. **存放损失的张量**

```python
    losses = torch.zeros(max_steps)
```

* 建立一个 shape=`[max_steps]` 的张量，依次存放每个 batch 的 loss。
* 最后会对它取平均，得到总体验证损失。

4. **循环计算验证集损失**

```python
    for k in range(max_steps):
        X, Y = get_batch(
            dataset=data,
            batch_size=batch_size,
            context_length=context_length,
            device=device,
        )
```

* 每次调用 `get_batch` 从验证数据里随机抽取一小段序列：

  * `X.shape = [B, T]`（输入 token ids）；
  * `Y.shape = [B, T]`（标签 token ids，即输入序列右移一位）。

```python
        logits = model(X)
```

* 前向传播得到 `logits`，形状 `[B, T, V]`：

  * `B`=batch\_size，
  * `T`=context\_length，
  * `V`=vocab\_size（词表大小）。

```python
        loss = cross_entropy(
            logits.view(-1, logits.size(-1)), 
            Y.view(-1)
        )
```

* 交叉熵计算 loss：

  * `logits.view(-1, V)`：把 `[B, T, V]` 展平成 `[B*T, V]`，把每个位置当作一次分类任务。
  * `Y.view(-1)`：标签展平成 `[B*T]`，对应每个时间步的“下一个 token”。
  * 返回的 `loss` 是一个标量（单个 float tensor），表示整个 batch 的平均 NLL（负对数似然）。

```python
        losses[k] = loss.item()
```

* `loss.item()` 取 Python float，存进 `losses[k]`。

5. **结束后恢复训练模式**

```python
    model.train()
```

* 把模型切换回训练模式（重新启用 dropout 等）。否则后面继续训练时行为会不对。

6. **返回平均验证损失**

```python
    return losses.mean()
```

* 对 `max_steps` 个 batch 的损失取平均。
* 这就是验证集的平均 loss（越低越好）。
* 在训练脚本里，它常用来计算 **perplexity**：`ppl = exp(val_loss)`。

In [7]:
@torch.no_grad()
def evaluate(model, data, context_length, batch_size, device, max_steps):
    """在验证集上评估模型"""
    model.eval()
    losses = torch.zeros(max_steps)
    for k in range(max_steps):
        X, Y = get_batch(
            dataset=data,
            batch_size=batch_size,
            context_length=context_length,
            device=device,
        )
        logits = model(X)
        loss = cross_entropy(logits.view(-1, logits.size(-1)), Y.view(-1))
        losses[k] = loss.item()
    model.train()
    return losses.mean()

## 6.3 训练流程


1. **函数入口与读取参数**

```python
def main():
    args = get_args()
```

* `get_args()`：解析命令行参数，把所有训练/模型/日志/性能相关配置收集到 `args`（如 `--batch_size`、`--d_model`、`--eval_interval` 等）。


2. **设备选择（CPU / CUDA / MPS）**

```python
# 设置设备
device = args.device
if "cuda" in device and not torch.cuda.is_available():
    print("CUDA not available, falling back to CPU")
    device = "cpu"
```

* 若想用 `cuda`，但环境无 GPU，就**降级**到 CPU。


3. **随机种子（可复现）**

```python
torch.manual_seed(1337)
if "cuda" in device:
    torch.cuda.manual_seed(1337)
```

* 作用：固定随机性（数据采样、权重初始化、dropout 等），让同配置的多次运行结果更接近。


4. **加载数据（内存映射，省内存）**

```python
print("Loading data...")
# 使用 mmap_mode='r' 进行内存高效加载
train_data = np.load(args.train_data_path, mmap_mode='r')
val_data = np.load(args.val_data_path, mmap_mode='r')
```

* `.npy` 里的 **token 序列** 通过 **内存映射**（`mmap_mode='r'`）读取：不把整块数据一次性载入内存，而是按需分页载入。


5. **构建模型并迁移到设备**

```python
# 初始化模型
model_args = {
    "vocab_size": args.vocab_size, "context_length": args.context_length,
    "d_model": args.d_model, "num_layers": args.num_layers,
    "num_heads": args.num_heads, "d_ff": args.d_ff,
    "rope_theta": args.rope_theta, "device": device,
}
model = TransformerLM(**model_args)
model.to(device)
```

* 把结构超参集中到 `model_args`，实例化 `TransformerLM`。
* `model.to(device)`：把所有参数/缓冲区迁移到指定设备。


6. **可选编译加速（`torch.compile`）**

```python
# 编译模型以提高性能
if args.compile:
    print("Compiling the model... (this may take a minute)")
    if "mps" in device:
        # MPS 后端支持有限
        model = torch.compile(model, backend="aot_eager")
    else:
        model = torch.compile(model)
```

* 目的：让 PyTorch 做图层面的融合/内核选择，**加速推理与训练**。


7. **初始化优化器（AdamW）**

```python
# 初始化优化器
AdamW = get_adamw_cls()
optimizer = AdamW(
    model.parameters(),
    lr=args.learning_rate,
    betas=(args.beta1, args.beta2),
    weight_decay=args.weight_decay
)
```

* `get_adamw_cls()`：拿到 AdamW 类。
* 关键超参：

  * `lr`：基础学习率（之后会被调度器动态覆盖）
  * `betas`：动量超参（`(β1, β2)`）
  * `weight_decay`：L2 正则（常对 `LayerNorm/bias/Embedding` 设置为 0）



8. **断点续训（Checkpoint）**

```python
# 检查点和续训逻辑
start_step = 0
if args.init_from == "resume":
    ckpt_path = os.path.join(args.out_dir, "ckpt.pt")
    if os.path.exists(ckpt_path):
        print(f"Resuming training from {ckpt_path}")
        start_step = load_checkpoint(src=ckpt_path, model=model, optimizer=optimizer)
    else:
        print(f"Checkpoint not found at {ckpt_path}, starting from scratch.")

os.makedirs(args.out_dir, exist_ok=True)
```

* 逻辑：

  * 若 `--init_from resume` 且文件存在，就**恢复模型+优化器+步数**，从 `start_step` 继续训练；
  * 否则从头开始。
* `os.makedirs(..., exist_ok=True)`：保证输出目录存在（首次训练会自动创建）。



9. **可选 W\&B 日志**

```python
# WandB 设置
if args.wandb_log:
    wandb.init(project=args.wandb_project, name=args.wandb_run_name, config=args)
```

* 作用：把本次实验的 **超参+指标** 同步到 Weights & Biases 的云端面板，便于对比可视化。
* `config=args`：把所有参数上传，方便回溯。



10. **训练主循环**

```python
print(f"Starting training for {args.max_steps} steps...")
t0 = time.time()
for step in range(start_step, args.max_steps):
```

* 从 `start_step`（可能是断点）跑到 `max_steps`。
* `t0` 用于统计日志时间间隔。

    1 **学习率调度（Warmup + Cosine）**
    
    ```python
    # 获取学习率
    lr = get_lr_cosine_schedule(
        it=step, max_learning_rate=args.learning_rate,
        min_learning_rate=args.min_learning_rate,
        warmup_iters=args.warmup_steps, cosine_cycle_iters=args.max_steps
    )
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    ```
    
    * 机制：
    
      * 预热 `warmup_iters`：从 0 → `max_learning_rate` **线性上升**。
      * 之后余弦退火：缓慢下降到 `min_learning_rate`。
    * 每步都把当前 `lr` 写回优化器的每个参数组，保证调度生效。
    
    2 **按间隔评估 + 打点 + 保存断点**
    
    ```python
    # 评估
    if step % args.eval_interval == 0 and step > 0:
        val_loss = evaluate(model, val_data, args.context_length, args.batch_size, device, args.eval_steps)
        print(f"Step {step}: Val Loss = {val_loss:.4f}, Val PPL = {torch.exp(val_loss):.2f}")
        if args.wandb_log:
            wandb.log({
                "eval/loss": val_loss,
                "eval/perplexity": torch.exp(val_loss),
            }, step=step)
        
        # 保存检查点
        ckpt_path = os.path.join(args.out_dir, "ckpt.pt")
        save_checkpoint(model=model, optimizer=optimizer, iteration=step, out=ckpt_path)
        print(f"Checkpoint saved to {ckpt_path}")
    ```
    
    * 功能：
    
      * 调 `evaluate(...)` 在验证集上抽取 `eval_steps` 个 batch，平均交叉熵为 `val_loss`。
      * 打印 `Val Loss` 和 `Val PPL=exp(loss)`；若开启 W\&B，`wandb.log` 同步两项指标。
      * 保存当前 checkpoint（便于中断恢复）。

    3 **取训练批（`get_batch`）**
    
    ```python
    # 获取训练数据
    X, Y = get_batch(
        dataset=train_data,
        batch_size=args.batch_size,
        context_length=args.context_length,
        device=device,
    )
    ```
    
    * 作用：从 memmap 的 token 序列里**随机抽**一段长度为 `T=context_length` 的片段，拼成一个批次：
    
      * `X: [B, T]` 输入；
      * `Y: [B, T]` 目标（通常是 `X` 右移一位，代表 next-token）。
    * 好处：无需 DataLoader 也能高速抽样；I/O 成本很低。
    
    4 **前向 → 损失 → 反向**
    
    ```python
    # 前向和后向传播
    logits = model(X)  # [B, T, V]
    loss = cross_entropy(
        logits.view(-1, logits.size(-1)),  # [B*T, V]
        Y.view(-1)                         # [B*T]
    )
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    ```
    
    * `logits: [B, T, V]`；把时间维展平后做分类交叉熵（每个位置都是 `V` 类分类任务）。
    * `zero_grad(set_to_none=True)`：把梯度置为 `None`（比置 0 省显存、略快）。
    * `loss.backward()`：反向求梯度。
    
    5 **梯度裁剪（防爆）**
    
    ```python
    # 梯度裁剪
    if args.grad_clip > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
    ```
    
    * 意义：限制梯度范数上限，**抑制梯度爆炸**，提升训练稳定性。
    
    6 **参数更新**
    
    ```python
    optimizer.step()
    ```
    
    * 用刚才的梯度按当前学习率更新参数。
    
    7 **训练日志与吞吐统计**
    
    ```python
    # 日志记录
    if step % args.log_interval == 0:
        t1 = time.time()
        dt = t1 - t0
        t0 = t1
        tokens_per_sec = (args.batch_size * args.context_length * args.log_interval) / dt
        print(f"Step {step:6d} | Loss: {loss.item():.4f} | LR: {lr:.2e} | Time: {dt*1000:.2f}ms | Tokens/sec: {tokens_per_sec:.0f}")
        if args.wandb_log:
            wandb.log({
                "train/loss": loss.item(),
                "train/lr": lr,
                "perf/tokens_per_sec": tokens_per_sec,
            }, step=step)
    ```
    
    * 打印：
    
      * `Loss`：当前 batch 的训练 loss；
      * `LR`：当前学习率（调度器产出）；
      * `Time`：本段 `log_interval` 步耗时（毫秒）；
      * `Tokens/sec`：**近似吞吐**（`B*T*log_interval / dt`）。

11. **训练完成**

```python
print("Training finished.")
```

* 训练循环退出（达到 `max_steps`），打印收尾信息。